# CASAS free-living, step 1: inventory
Reads the unzipped data/, labeled_data/ and floorplans/ folders from Drive (no download) and profiles every home. Nothing is filtered yet.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = "https://github.com/<user>/<repo>.git"   # TODO: your casas_audit repo
BRANCH = "main"
PROJECT_ROOT_DIR = "/content/drive/MyDrive/02_research/hdeg3"        # outputs/ and cache/ go here
DATASET_FOLDER = "/content/drive/MyDrive/02_research/hdeg3/data"     # TODO: folder holding data/, labeled_data/, floorplans/ (found up to 3 levels down)
SETS = []          # e.g. ["inventory.workers=4"]; labels + floorplans only (reuse the data inventory): ["inventory.parts=[labeled,floorplans]"]
RUN_INVENTORY = False   # step 1 already done (outputs on Drive)
RUN_SELECT = False      # step 2 already done (outputs on Drive)
RUN_AUDIT = False       # step 3 already done (10-min step cache is on Drive)
AUDIT_SETS = []         # e.g. ["audit.tiers=[core_single,multi_resident]", "audit.workers=4"]
SELECT_SETS = []   # e.g. ["selection.min_window_active_days=84", "selection.min_rooms_with_2plus=4"]
config_override = "{}"
RUN_STAGE_E = True      # step 4: v2 = hour calibration + hierarchical autoencoder -> outputs/casas_stage_e_v2 (GPU; resumable)
STAGE_E_EXTRA = ""      # "--quick" = smoke test on 3 homes (separate output folder, not for interpretation)
STAGE_E_SETS = []       # e.g. ["stage_e.homes=[tm008,tm010]"] to run a subset; finished results are reused

In [ ]:
import os, shutil, multiprocessing
print("CPUs:", multiprocessing.cpu_count())
repo = "/content/casas_audit"
if os.path.exists(repo): shutil.rmtree(repo)
!git clone -b $BRANCH $REPO_URL $repo
%cd $repo
!pip install -q -r requirements.txt
open("configs/config.yaml", "w").write(config_override)

In [ ]:
if RUN_INVENTORY:
    sets = " ".join(f"--set '{s}'" for s in SETS)
    !python -m run_inventory --project_root_dir $PROJECT_ROOT_DIR --dataset_folder $DATASET_FOLDER $sets
if RUN_SELECT:
    ssets = " ".join(f"--set '{s}'" for s in SELECT_SETS)
    !python -m run_select --project_root_dir $PROJECT_ROOT_DIR $ssets
if RUN_AUDIT:
    asets = " ".join(f"--set '{s}'" for s in AUDIT_SETS)
    !python -W ignore -m run_audit --project_root_dir $PROJECT_ROOT_DIR --dataset_folder $DATASET_FOLDER $asets
if RUN_STAGE_E:
    esets = " ".join(f"--set '{s}'" for s in STAGE_E_SETS)
    !python -W ignore -m run_stage_e --project_root_dir $PROJECT_ROOT_DIR $STAGE_E_EXTRA $esets

In [ ]:
from IPython.display import Markdown, display
for f in ("casas_inventory/inventory_report.md", "casas_selection/selection_report.md", "casas_audit/audit_report.md", "casas_stage_e/stage_e_report.md"):
    p = f"{PROJECT_ROOT_DIR}/outputs/{f}"
    if os.path.exists(p): display(Markdown(open(p).read()))